In [1]:
from dataclasses import dataclass
from enum import Enum
import random
import time
import uuid
from typing import Any, Callable, Dict, List, Optional


# ==========================================
# 1. 归一化异常体系与统一契约 (ErrorKind & ToolError & ToolResult)
# ==========================================
class ErrorKind(Enum):
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"
    RATE_LIMITED = "rate_limited"
    TIMEOUT = "timeout"
    OVERLOADED = "overloaded"
    CANCELLED = "cancelled"
    INTERNAL = "internal"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool
    status_code: Optional[int] = None
    retry_after: Optional[float] = None


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None
    attempts: int = 0
    latency_ms: float = 0.0


def normalize_error(
    status_code: Optional[int] = None,
    exc: Optional[Exception] = None,
    retry_after: Optional[float] = None,
    is_bulkhead_rejected: bool = False,
    is_circuit_open: bool = False,
) -> ToolError:
    """将 Raw Error / 熔断 / 隔舱信号归一化映射为标准 ToolError"""
    if is_circuit_open:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="CircuitBreaker is OPEN",
            retryable=False,
        )

    if is_bulkhead_rejected:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="Bulkhead capacity full",
            retryable=False,  # 避免对过载系统发起重试风暴
            status_code=503,
        )

    if exc is not None:
        if isinstance(exc, TimeoutError):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=str(exc) or "Request execution timed out",
                retryable=True,
            )
        return ToolError(
            kind=ErrorKind.INTERNAL,
            message=f"Internal exception: {type(exc).__name__} - {str(exc)}",
            retryable=False,
        )

    if status_code is not None:
        if status_code == 429:
            return ToolError(
                kind=ErrorKind.RATE_LIMITED,
                message="HTTP 429 Too Many Requests",
                retryable=True,
                status_code=429,
                retry_after=retry_after or 0.1,
            )
        if status_code in (500, 502, 503, 504):
            return ToolError(
                kind=ErrorKind.RETRYABLE,
                message=f"HTTP {status_code} Server Error",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (408,):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=f"HTTP {status_code} Request Timeout",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (400, 401, 403, 404, 422):
            return ToolError(
                kind=ErrorKind.NON_RETRYABLE,
                message=f"HTTP {status_code} Client Error",
                retryable=False,
                status_code=status_code,
            )

    return ToolError(
        kind=ErrorKind.INTERNAL,
        message=f"Unknown raw error (status={status_code})",
        retryable=False,
        status_code=status_code,
    )


# ==========================================
# 2. 基础组件与配置 (CircuitBreaker & Bulkhead & Registry)
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    idempotency_required: bool
    breaker: CircuitBreaker
    bulkhead: Bulkhead


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


# ==========================================
# 3. 强类型 ToolResult 输出的 ToolRuntime
# ==========================================
class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> ToolResult:
        start_time = time.perf_counter()

        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        if config.idempotency_required and "idempotency_key" not in args:
            args["idempotency_key"] = f"idempotent-{uuid.uuid4().hex[:8]}"

        if deadline is None:
            deadline = time.time() + 10.0

        # Breaker Gate 检查
        if not config.breaker.can_call():
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                data=None,
                error=normalize_error(is_circuit_open=True),
                attempts=0,
                latency_ms=elapsed_ms,
            )

        retry_count = 0

        while True:
            # Bulkhead Acquire 检查
            if not config.bulkhead.try_acquire():
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=normalize_error(is_bulkhead_rejected=True),
                    attempts=retry_count,
                    latency_ms=elapsed_ms,
                )

            raw_result = None
            caught_exc = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                caught_exc = e
            finally:
                config.bulkhead.release()

            # 提取状态信息
            status_code = None
            retry_after = None
            if isinstance(raw_result, dict):
                status_code = raw_result.get("status")
                retry_after = raw_result.get("retry_after")

            # 错误归一化判断
            if caught_exc is not None or (status_code and status_code != 200):
                tool_error = normalize_error(
                    status_code=status_code,
                    exc=caught_exc,
                    retry_after=retry_after,
                )
            else:
                tool_error = None

            # 1. 成功出口
            if tool_error is None:
                config.breaker.record_success()
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=True,
                    tool_name=tool_name,
                    data=raw_result,
                    error=None,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 2. 失败处理路径
            config.breaker.record_failure()

            # 不可重试错误出口
            if not tool_error.retryable:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=raw_result,
                    error=tool_error,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 超过最大重试次数出口
            if retry_count >= config.max_retries:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=raw_result,
                    error=tool_error,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 退避策略 (Backoff)
            if tool_error.kind == ErrorKind.RATE_LIMITED and tool_error.retry_after:
                backoff = tool_error.retry_after
            else:
                backoff = config.base_delay * (2 ** retry_count) + random.uniform(0.0, 0.01)

            now = time.time()
            # 超出 Deadline 限制出口
            if (deadline - now) < (backoff + config.timeout):
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                deadline_error = ToolError(
                    kind=ErrorKind.TIMEOUT,
                    message="Deadline exceeded before next retry backoff",
                    retryable=False,
                )
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=deadline_error,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            time.sleep(backoff)
            retry_count += 1


# ==========================================
# 4. Mock 工具与场景验证
# ==========================================
class SequenceMockTool:
    def __init__(self, sequence: List[Any]):
        self.sequence = sequence

    def __call__(self, args: Dict[str, Any]) -> Dict[str, Any]:
        item = self.sequence.pop(0) if self.sequence else 200
        if isinstance(item, Exception):
            raise item
        if isinstance(item, dict):
            return item
        return {"status": item, "data": f"Response data for {args.get('q', 'action')}"}


if __name__ == "__main__":
    registry = ToolRegistry()

    def make_config(name: str, max_retries: int = 1) -> ToolConfig:
        return ToolConfig(
            name=name,
            max_retries=max_retries,
            base_delay=0.01,
            timeout=1.0,
            idempotency_required=True,
            breaker=CircuitBreaker(),
            bulkhead=Bulkhead(capacity=2),
        )

    runtime = ToolRuntime(registry)

    print("=== Scenario 1: Search success ===")
    search_mock = SequenceMockTool(sequence=[200])
    registry.register(make_config("search_tool"), search_mock)

    res1 = runtime.execute("search_tool", {"q": "langgraph"})
    print(f"ToolResult: {res1}")
    print(f"ok: {res1.ok} (Expected: True)")
    print(f"data: {res1.data}")
    print(f"error: {res1.error} (Expected: None)")
    print(f"latency_ms: {res1.latency_ms:.2f}ms\n")

    print("=== Scenario 2: HR 400 ===")
    hr_400_mock = SequenceMockTool(sequence=[400])
    registry.register(make_config("hr_400_tool", max_retries=2), hr_400_mock)

    res2 = runtime.execute("hr_400_tool", {})
    print(f"ToolResult: {res2}")
    print(f"ok: {res2.ok} (Expected: False)")
    print(f"error.kind: {res2.error.kind} (Expected: ErrorKind.NON_RETRYABLE)\n")

    print("=== Scenario 3: HR 503 -> retry -> 200 ===")
    hr_503_mock = SequenceMockTool(sequence=[503, 200])
    registry.register(make_config("hr_503_tool", max_retries=2), hr_503_mock)

    res3 = runtime.execute("hr_503_tool", {})
    print(f"ToolResult: {res3}")
    print(f"ok: {res3.ok} (Expected: True)")
    print(f"attempts: {res3.attempts} (Expected: 2)")
    print(f"error: {res3.error} (Expected: None)\n")

    print("=== Scenario 4: HR RuntimeError ===")
    hr_crash_mock = SequenceMockTool(sequence=[RuntimeError("Internal Error")])
    registry.register(make_config("hr_crash_tool", max_retries=2), hr_crash_mock)

    res4 = runtime.execute("hr_crash_tool", {})
    print(f"ToolResult: {res4}")
    print(f"ok: {res4.ok} (Expected: False)")
    print(f"error.kind: {res4.error.kind} (Expected: ErrorKind.INTERNAL)")

=== Scenario 1: 429 -> RATE_LIMITED (包含 retry_after 退避并重试成功) ===
Status: SUCCESS | Attempts: 2

=== Scenario 2: 503 -> RETRYABLE (识别为可重试错误，次轮成功) ===
Status: SUCCESS | Attempts: 2

=== Scenario 3: 400 -> NON_RETRYABLE (客户端错误，禁止重试直接 FAST FAIL/FATAL) ===
Status: FATAL_ERROR | ErrorKind: non_retryable | Attempts: 1 (Expected: 1)

=== Scenario 4: RuntimeError -> INTERNAL (捕获未处理的代码异常，归一化为不可重试内部错误) ===
Status: FATAL_ERROR | ErrorKind: internal | Attempts: 1 (Expected: 1)

